In [1]:
pip install faker

Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
E-Commerce Benchmark Dataset Generator for CockroachDB
Generates realistic synthetic data matching the exact schema.
Run: python generate_data.py
"""

import csv
import random
from datetime import datetime, timedelta
from decimal import Decimal
from faker import Faker
from collections import defaultdict

# Initialize Faker
fake = Faker()
Faker.seed(42)  # Reproducible data
random.seed(42)

# Output directory
OUTPUT_DIR = "ecommerce_data"

# Create output directory
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("🔧 Starting dataset generation...")

# ============================================
# 1. CATEGORIES (50 rows)
# ============================================
print("📦 Generating categories...")

CATEGORY_NAMES = [
    "Electronics", "Computers", "Smartphones", "Tablets", "Laptops",
    "Cameras", "Audio", "Headphones", "Speakers", "Wearables",
    "Home & Kitchen", "Furniture", "Appliances", "Cookware", "Bedding",
    "Clothing", "Men's Clothing", "Women's Clothing", "Kids' Clothing", "Shoes",
    "Sports & Outdoors", "Exercise Equipment", "Camping", "Cycling", "Sports Apparel",
    "Books", "Fiction", "Non-Fiction", "Educational", "Comics",
    "Toys & Games", "Board Games", "Action Figures", "Puzzles", "Educational Toys",
    "Beauty & Personal Care", "Skincare", "Makeup", "Hair Care", "Fragrances",
    "Automotive", "Car Accessories", "Tools", "Motorcycle", "Tires",
    "Garden & Outdoor", "Plants", "Garden Tools", "Outdoor Furniture", "Grills"
]

categories = []
for i in range(50):
    category_id = i + 1
    name = CATEGORY_NAMES[i]
    
    # Create hierarchy: 20% of categories have a parent
    parent_id = None
    if i > 10 and random.random() < 0.4:
        parent_id = random.randint(1, min(i, 15))  # Parent from earlier categories
    
    categories.append({
        'category_id': category_id,
        'name': name,
        'parent_id': parent_id
    })

# Write categories CSV
with open(f'{OUTPUT_DIR}/categories.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['category_id', 'name', 'parent_id'])
    writer.writeheader()
    writer.writerows(categories)

print(f"  ✅ Generated {len(categories)} categories")

# ============================================
# 2. PRODUCTS (10,000 rows)
# ============================================
print("📦 Generating products...")

# Hot categories (20% of categories get 60% of products)
hot_categories = random.sample(range(1, 51), 10)
category_weights = [3 if i in hot_categories else 1 for i in range(1, 51)]

products = []
for i in range(10000):
    product_id = i + 1
    
    # Weighted category selection (creates realistic skew)
    category_id = random.choices(range(1, 51), weights=category_weights, k=1)[0]
    
    # Generate realistic product name
    adjective = random.choice(['Premium', 'Professional', 'Deluxe', 'Ultra', 'Classic', 
                               'Modern', 'Advanced', 'Essential', 'Pro', 'Smart'])
    product_type = random.choice(['Widget', 'Gadget', 'Device', 'Tool', 'Kit', 
                                  'Set', 'System', 'Accessory', 'Bundle', 'Product'])
    name = f"{adjective} {product_type} {fake.color_name()}"
    
    # Realistic price distribution
    price = round(random.uniform(9.99, 999.99), 2)
    
    # Stock quantity (realistic retail inventory)
    stock_qty = random.choices(
        [0, random.randint(1, 10), random.randint(10, 100), random.randint(100, 1000)],
        weights=[5, 20, 50, 25],  # 5% out of stock, most have moderate stock
        k=1
    )[0]
    
    description = fake.text(max_nb_chars=200)
    created_at = fake.date_time_between(start_date='-2y', end_date='now')
    
    products.append({
        'product_id': product_id,
        'name': name,
        'category_id': category_id,
        'price': price,
        'stock_qty': stock_qty,
        'description': description,
        'created_at': created_at.strftime('%Y-%m-%d %H:%M:%S')
    })

# Write products CSV
with open(f'{OUTPUT_DIR}/products.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['product_id', 'name', 'category_id', 'price', 
                                            'stock_qty', 'description', 'created_at'])
    writer.writeheader()
    writer.writerows(products)

print(f"  ✅ Generated {len(products)} products")

# ============================================
# 3. USERS (5,000 rows)
# ============================================
print("👥 Generating users...")

users = []
emails_set = set()  # Ensure unique emails

for i in range(5000):
    user_id = i + 1
    
    # Generate unique email
    while True:
        email = fake.unique.email()
        if email not in emails_set:
            emails_set.add(email)
            break
    
    name = fake.name()
    created_at = fake.date_time_between(start_date='-3y', end_date='now')
    
    users.append({
        'user_id': user_id,
        'email': email,
        'name': name,
        'created_at': created_at.strftime('%Y-%m-%d %H:%M:%S')
    })

# Write users CSV
with open(f'{OUTPUT_DIR}/users.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['user_id', 'email', 'name', 'created_at'])
    writer.writeheader()
    writer.writerows(users)

print(f"  ✅ Generated {len(users)} users")

# ============================================
# 4. CART_ITEMS (8,000 rows)
# ============================================
print("🛒 Generating cart items...")

# Hot products (10% of products appear in 50% of carts)
hot_products = random.sample(range(1, 10001), 1000)
product_weights = [2 if i in hot_products else 1 for i in range(1, 10001)]

cart_items = []
cart_pairs = set()  # Track (user_id, product_id) for uniqueness

attempt = 0
max_attempts = 20000

while len(cart_items) < 8000 and attempt < max_attempts:
    attempt += 1
    
    user_id = random.randint(1, 5000)
    product_id = random.choices(range(1, 10001), weights=product_weights, k=1)[0]
    
    # Enforce unique constraint
    if (user_id, product_id) in cart_pairs:
        continue
    
    cart_pairs.add((user_id, product_id))
    
    cart_item_id = len(cart_items) + 1
    quantity = random.randint(1, 5)
    added_at = fake.date_time_between(start_date='-6m', end_date='now')
    
    cart_items.append({
        'cart_item_id': cart_item_id,
        'user_id': user_id,
        'product_id': product_id,
        'quantity': quantity,
        'added_at': added_at.strftime('%Y-%m-%d %H:%M:%S')
    })

# Write cart_items CSV
with open(f'{OUTPUT_DIR}/cart_items.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['cart_item_id', 'user_id', 'product_id', 
                                            'quantity', 'added_at'])
    writer.writeheader()
    writer.writerows(cart_items)

print(f"  ✅ Generated {len(cart_items)} cart items")

# ============================================
# 5. ORDERS (3,000 rows)
# ============================================
print("📦 Generating orders...")

ORDER_STATUSES = ['completed', 'pending', 'cancelled']
STATUS_WEIGHTS = [70, 20, 10]  # 70% completed, 20% pending, 10% cancelled

orders = []
for i in range(3000):
    order_id = i + 1
    user_id = random.randint(1, 5000)
    status = random.choices(ORDER_STATUSES, weights=STATUS_WEIGHTS, k=1)[0]
    created_at = fake.date_time_between(start_date='-1y', end_date='now')
    
    # total_amount will be calculated after order_items are generated
    orders.append({
        'order_id': order_id,
        'user_id': user_id,
        'status': status,
        'total_amount': 0,  # Placeholder
        'created_at': created_at.strftime('%Y-%m-%d %H:%M:%S')
    })

print(f"  ✅ Generated {len(orders)} orders")

# ============================================
# 6. ORDER_ITEMS (9,000 rows)
# ============================================
print("📦 Generating order items...")

# Map to track order totals
order_totals = defaultdict(Decimal)

# Create product price lookup
product_price_map = {p['product_id']: Decimal(str(p['price'])) for p in products}

order_items = []
for i in range(9000):
    order_item_id = i + 1
    order_id = random.randint(1, 3000)
    product_id = random.choices(range(1, 10001), weights=product_weights, k=1)[0]
    quantity = random.randint(1, 3)
    
    # Use product's current price (realistic: price at time of order)
    unit_price = product_price_map[product_id]
    
    # Add to order total
    order_totals[order_id] += unit_price * quantity
    
    order_items.append({
        'order_item_id': order_item_id,
        'order_id': order_id,
        'product_id': product_id,
        'quantity': quantity,
        'unit_price': float(unit_price)
    })

# Write order_items CSV
with open(f'{OUTPUT_DIR}/order_items.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['order_item_id', 'order_id', 'product_id', 
                                            'quantity', 'unit_price'])
    writer.writeheader()
    writer.writerows(order_items)

print(f"  ✅ Generated {len(order_items)} order items")

# ============================================
# Update order totals
# ============================================
print("💰 Calculating order totals...")

for order in orders:
    order['total_amount'] = float(order_totals.get(order['order_id'], Decimal('0')))

# Write orders CSV
with open(f'{OUTPUT_DIR}/orders.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['order_id', 'user_id', 'status', 
                                            'total_amount', 'created_at'])
    writer.writeheader()
    writer.writerows(orders)

print(f"  ✅ Updated order totals")

# ============================================
# VALIDATION CHECKS
# ============================================
print("\n🔍 Running validation checks...")

errors = []

# Check row counts
if len(categories) != 50:
    errors.append(f"❌ Categories: expected 50, got {len(categories)}")
if len(products) != 10000:
    errors.append(f"❌ Products: expected 10000, got {len(products)}")
if len(users) != 5000:
    errors.append(f"❌ Users: expected 5000, got {len(users)}")
if len(cart_items) != 8000:
    errors.append(f"❌ Cart items: expected 8000, got {len(cart_items)}")
if len(orders) != 3000:
    errors.append(f"❌ Orders: expected 3000, got {len(orders)}")
if len(order_items) != 9000:
    errors.append(f"❌ Order items: expected 9000, got {len(order_items)}")

# Check unique emails
if len(emails_set) != 5000:
    errors.append(f"❌ Emails not unique: {len(emails_set)} unique emails")

# Check cart_items uniqueness
if len(cart_pairs) != len(cart_items):
    errors.append(f"❌ Cart items (user_id, product_id) not unique")

# Check foreign keys
product_ids = {p['product_id'] for p in products}
user_ids = {u['user_id'] for u in users}
category_ids = {c['category_id'] for c in categories}
order_ids = {o['order_id'] for o in orders}

for item in cart_items:
    if item['user_id'] not in user_ids:
        errors.append(f"❌ Invalid user_id in cart_items: {item['user_id']}")
        break
    if item['product_id'] not in product_ids:
        errors.append(f"❌ Invalid product_id in cart_items: {item['product_id']}")
        break

for item in order_items:
    if item['order_id'] not in order_ids:
        errors.append(f"❌ Invalid order_id in order_items: {item['order_id']}")
        break
    if item['product_id'] not in product_ids:
        errors.append(f"❌ Invalid product_id in order_items: {item['product_id']}")
        break

# Check orders have valid totals
orders_without_items = [o for o in orders if o['order_id'] not in order_totals]
if len(orders_without_items) > 0:
    print(f"  ⚠️  {len(orders_without_items)} orders have no items (total_amount = 0)")

if len(errors) == 0:
    print("  ✅ All validation checks passed!")
else:
    print("  ❌ Validation errors found:")
    for error in errors:
        print(f"     {error}")

print(f"\n✨ Dataset generation complete!")
print(f"📁 Files saved to: {OUTPUT_DIR}/")
print(f"\nGenerated files:")
print(f"  • categories.csv (50 rows)")
print(f"  • products.csv (10,000 rows)")
print(f"  • users.csv (5,000 rows)")
print(f"  • cart_items.csv (8,000 rows)")
print(f"  • orders.csv (3,000 rows)")
print(f"  • order_items.csv (9,000 rows)")

🔧 Starting dataset generation...
📦 Generating categories...
  ✅ Generated 50 categories
📦 Generating products...
  ✅ Generated 10000 products
👥 Generating users...
  ✅ Generated 5000 users
🛒 Generating cart items...
  ✅ Generated 8000 cart items
📦 Generating orders...
  ✅ Generated 3000 orders
📦 Generating order items...
  ✅ Generated 9000 order items
💰 Calculating order totals...
  ✅ Updated order totals

🔍 Running validation checks...
  ⚠️  156 orders have no items (total_amount = 0)
  ✅ All validation checks passed!

✨ Dataset generation complete!
📁 Files saved to: ecommerce_data/

Generated files:
  • categories.csv (50 rows)
  • products.csv (10,000 rows)
  • users.csv (5,000 rows)
  • cart_items.csv (8,000 rows)
  • orders.csv (3,000 rows)
  • order_items.csv (9,000 rows)
